In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# ==========================================
# CONFIGURACIÓN GENERAL
# ==========================================
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Función auxiliar para convertir textos como "0,78" a números decimales 0.78
def texto_a_numero(texto):
    if not texto:
        return None
    try:
        # Quitamos espacios, símbolos de % y cambiamos la coma por punto
        texto_limpio = texto.strip().replace('%', '').replace(',', '.')
        return float(texto_limpio)
    except ValueError:
        return None

# Listas para guardar los datos de ambas páginas
datos_embalses = []
datos_caudales = []

# ==========================================
# PARTE 1: SCRAPING DE EMBALSES
# ==========================================
url_embalses = "https://saih.chj.es/embalses"
response_embalses = requests.get(url_embalses, headers=headers)

if response_embalses.status_code == 200:
    print("Página de embalses descargada. Extrayendo datos...")
    soup_embalses = BeautifulSoup(response_embalses.content, 'html.parser')
    filas_embalses = soup_embalses.find_all('tr')
    
    for fila in filas_embalses:
        columnas = fila.find_all('td')
        if len(columnas) > 1:
            if 'text-nowrap' in columnas[0].get('class', []):
                nombre = columnas[0].text.strip()
                if "TOTAL" in nombre.upper() or nombre == "":
                    continue
                
                volumen = texto_a_numero(columnas[1].text)
                
                contenedor_progreso = fila.find('div', class_='progress')
                if contenedor_progreso:
                    porcentaje = texto_a_numero(contenedor_progreso.text)
                else:
                    porcentaje = texto_a_numero(columnas[-1].text)
                
                datos_embalses.append({
                    'Nombre': nombre,
                    'Volumen embalsado (hm3)': volumen,
                    '% Volumen sobre nivel máximo': porcentaje
                })
else:
    print(f"Error al acceder a Embalses: {response_embalses.status_code}")

# ==========================================
# PARTE 2: SCRAPING DE CAUDALES (AFOROS)
# ==========================================
url_caudales = "https://saih.chj.es/aforos"
response_caudales = requests.get(url_caudales, headers=headers)

if response_caudales.status_code == 200:
    print("Página de caudales descargada. Extrayendo datos...")
    soup_caudales = BeautifulSoup(response_caudales.content, 'html.parser')
    filas_caudales = soup_caudales.find_all('tr')
    
    for fila in filas_caudales:
        columnas = fila.find_all('td')
        
        # Comprobamos que tenga al menos 6 columnas (para cubrir hasta el umbral rojo)
        if len(columnas) >= 6:
            # Filtramos por la clase 'text-start' que has identificado para los nombres
            if 'text-start' in columnas[0].get('class', []):
                punto = columnas[0].text.strip()
                variable = columnas[1].text.strip()
                
                # Extraemos y convertimos los valores numéricos basándonos en los índices que encontraste
                ultimo_valor = texto_a_numero(columnas[2].text)
                umbral_amarillo = texto_a_numero(columnas[3].text)
                umbral_naranja = texto_a_numero(columnas[4].text)
                umbral_rojo = texto_a_numero(columnas[5].text)
                
                datos_caudales.append({
                    'Punto': punto,
                    'Variable': variable,
                    'Último valor (m3/s)': ultimo_valor,
                    'Umbral Amarillo (m3/s)': umbral_amarillo,
                    'Umbral Naranja (m3/s)': umbral_naranja,
                    'Umbral Rojo (m3/s)': umbral_rojo
                })
else:
    print(f"Error al acceder a Caudales: {response_caudales.status_code}")

# ==========================================
# PARTE 3: GUARDAR TODO EN UN ÚNICO EXCEL
# ==========================================
if datos_embalses or datos_caudales:
    nombre_archivo = "datos_hidrograficos_chj.xlsx"
    
    # Creamos los DataFrames
    df_embalses = pd.DataFrame(datos_embalses)
    df_caudales = pd.DataFrame(datos_caudales)
    
    # Abrimos el ExcelWriter
    with pd.ExcelWriter(nombre_archivo, engine='openpyxl') as writer:
        if not df_embalses.empty:
            df_embalses.to_excel(writer, sheet_name='embalses', index=False)
            print("- Hoja 'embalses' creada.")
            
        if not df_caudales.empty:
            df_caudales.to_excel(writer, sheet_name='caudales', index=False)
            print("- Hoja 'caudales' creada.")
            
    print(f"\n¡Proceso terminado! Todos los datos se han guardado en el archivo: {nombre_archivo}")
else:
    print("\nNo se pudo extraer ningún dato de las páginas.")

Página de embalses descargada. Extrayendo datos...
Página de caudales descargada. Extrayendo datos...
- Hoja 'embalses' creada.
- Hoja 'caudales' creada.

¡Proceso terminado! Todos los datos se han guardado en el archivo: datos_hidrograficos_chj.xlsx
